In [28]:
import torch
import os
from dataloaders import load_data
from decompositions.CNNFC import CNNFC
from decompositions.MLP import NeuralSVD_MLP
from utilitys import create_output_dir,save_loss_to_csv


In [ ]:
# def process(p, save):
#     import numpy as np
#     from PIL import Image

#     # Read TIFF image
#     img = Image.open(p)

#     # Convert to grayscale
#     gray_img = img.convert('L')

#     # Convert to numpy array
#     gray_array = np.array(gray_img)

#     # Save as npy file (using the provided save path)
#     np.save(save, gray_array)
    
#     return gray_array  # Optional: return array for further processing


# # Example usage
# process(r"E:\SVD\data\4.1.02.tiff", r"E:\SVD\grey_data\2")


In [ ]:
def train(model, data, optimizer, epochs, save_path, use_k_loop=None):
    """
    Compatible training function that supports two modes:
    1. K-loop mode: train sequentially for each k value (e.g., CNNFC model)
    2. Direct mode: train directly without k loop (e.g., DIP model)
    
    Args:
    model: the model
    data: training data
    optimizer: optimizer
    epochs: number of training epochs
    save_path: path to save results
    use_k_loop: whether to use k-loop mode, automatically detected if None
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(device)
    model.to(device).train()
    data = data.to(device)
    
    
    if use_k_loop:
        # K-loop mode - suitable for CNNFC and similar models
        k_losses = {k: {'mse': [], 'ortho': [], 'total': []} for k in range(1, model.k + 1)}
        
        for k in range(1, model.k + 1):
            print(f"\nStart to train k={k}")
            for epoch in range(epochs):
                optimizer.zero_grad()
                output = model.forward(data, k)
                loss_recon, loss_ortho, total_loss = model.loss(data, output)
                
                # Record losses
                k_losses[k]['mse'].append(loss_recon.item())
                k_losses[k]['ortho'].append(loss_ortho.item())
                k_losses[k]['total'].append(total_loss.item())
                
                total_loss.backward()
                optimizer.step()
                
                if epoch % 100 == 0:
                    print(f'Epoch: {epoch:3d}/{epochs}, '
                          f'MSE_loss: {loss_recon.item():.6f}, Ortho_loss {loss_ortho.item():.6f}, '
                          f'Total_loss: {total_loss.item():.6f}')
        
        save_loss_to_csv(k_losses, save_path)
        print("Everything is done! Saved to ", save_path)
        return model, k_losses
    
    else:
        # Direct training mode
        losses = {'mse': [], 'ortho': [], 'total': []}
        
        print(f"\nStart to train for {epochs} epochs")
        for epoch in range(epochs):
            optimizer.zero_grad()
            output = model.forward(data)
            loss_recon, loss_ortho, total_loss = model.loss(data, output)
            
            # Record losses
            losses['mse'].append(loss_recon.item())
            losses['ortho'].append(loss_ortho.item())
            losses['total'].append(total_loss.item())
            
            total_loss.backward()
            optimizer.step()
            
            if epoch % 100 == 0:
                print(f'Epoch: {epoch:3d}/{epochs}, '
                      f'MSE Loss: {loss_recon.item():.6f}, Orthogonality Loss: {loss_ortho.item():.6f}, '
                      f'Total Loss: {total_loss.item():.6f}')
        
        # Save model info if available
        if hasattr(model, 'info'):
            with open(os.path.join(save_path, 'model_info.txt'), 'w', encoding='utf-8') as f:
                f.write(model.info())
        
        save_loss_to_csv(losses, save_path)
        print("All operations completed! Save path:", save_path)
        return model, losses


In [32]:
data, input_size, data_name = load_data(4)
print(input_size)

torch.Size([256, 256])


In [33]:
k=64
model1 = CNNFC(k=k, base_channels=1,input_size=input_size)
optimizer1 = torch.optim.Adam(model1.parameters(), lr=0.001)
model2 = NeuralSVD_MLP(k=k, input_size=input_size)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001)

In [34]:
save_path, _ = create_output_dir(model1.__class__.__name__, data_name, k)

Created output directory: outputs\CNNFC_4_k64_0914


In [ ]:
model1,k_losses1 = train(model1, data, optimizer1, 100 ,save_path,use_k_loop=1)
model2,k_losses2 = train(model2, data, optimizer2, 2000 ,save_path,use_k_loop=1)

cuda

Start to train k=1
Epoch:   0/100, MSE_loss: 21237.800781, Ortho_loss 0.000000, Total_loss: 21237.800781

Start to train k=2
Epoch:   0/100, MSE_loss: 2846.484863, Ortho_loss 1.386294, Total_loss: 5691.705078

Start to train k=3
Epoch:   0/100, MSE_loss: 1991.072998, Ortho_loss 3.583519, Total_loss: 6827.184570

Start to train k=4
Epoch:   0/100, MSE_loss: 1475.827759, Ortho_loss 6.356108, Total_loss: 7787.468262

Start to train k=5
Epoch:   0/100, MSE_loss: 1257.363037, Ortho_loss 9.574984, Total_loss: 8825.892578

Start to train k=6
Epoch:   0/100, MSE_loss: 1131.844238, Ortho_loss 13.158503, Total_loss: 9831.503906

Start to train k=7
Epoch:   0/100, MSE_loss: 1016.134460, Ortho_loss 17.050323, Total_loss: 10731.544922

Start to train k=8
Epoch:   0/100, MSE_loss: 907.367249, Ortho_loss 21.209206, Total_loss: 11529.636719

Start to train k=9
Epoch:   0/100, MSE_loss: 821.597534, Ortho_loss 25.603655, Total_loss: 12265.346680

Start to train k=10
Epoch:   0/100, MSE_loss: 743.6

In [ ]:
# from visualizations import visualize_results

import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import torch.nn.functional as F


def visualize_results(data, model1, model2, k, path):
    """
    Visualization of results:
    1. Save reconstruction images (reconstruction.png)
    2. Save orthogonality plots (orthogonality.png)
    3. Save MSE/RelErr/OrthErr to mse.txt

    Args:
    data: input data (2D/3D Tensor)
    model1, model2: trained models
    k: rank
    path: save path
    """
    os.makedirs(path, exist_ok=True)

    # ===== Prepare original data =====
    if data.dim() == 3:  # (batch, H, W) -> take the first one
        data = data[0]
    data_np = data.cpu().detach().numpy()

    # ==========================================================
    # 1. Reconstruction + MSE
    # ==========================================================
    plt.figure(figsize=(20, 5))

    # Original
    plt.subplot(141)
    plt.imshow(data_np, cmap="gray")
    plt.axis("off")

    # ---- SVD reconstruction ----
    U, S, Vh = torch.linalg.svd(data, full_matrices=False)
    svd_recon = (U[:, :k] @ torch.diag(S[:k])) @ Vh[:k, :]
    svd_recon_np = svd_recon.cpu().detach().numpy()
    svd_mse = np.mean((data_np - svd_recon_np) ** 2) / 65025
    plt.subplot(142)
    plt.imshow(svd_recon_np, cmap="gray")
    plt.axis("off")

    # ---- Model1 reconstruction ----
    with torch.no_grad():
        out1 = model1(data.unsqueeze(0).to(next(model1.parameters()).device), k)
        recon1 = out1["X_rec"][0].cpu().detach().numpy()
    model1_mse = np.mean((data_np - recon1) ** 2) / 65025
    plt.subplot(143)
    plt.imshow(recon1, cmap="gray")
    plt.axis("off")

    # ---- Model2 reconstruction ----
    with torch.no_grad():
        out2 = model2(data.unsqueeze(0).to(next(model2.parameters()).device), k)
        recon2 = out2["X_rec"][0].cpu().detach().numpy()
    model2_mse = np.mean((data_np - recon2) ** 2) / 65025
    plt.subplot(144)
    plt.imshow(recon2, cmap="gray")
    plt.axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(path, "reconstruction.png"))
    plt.close()

    # ==========================================================
    # 2. Orthogonality
    # ==========================================================
    plt.figure(figsize=(25, 5))

    # ---- Model1 U/V ----
    U1, V1 = out1["U"][0], out1["V"][0]
    U1n = U1 / torch.norm(U1, dim=0, keepdim=True)
    V1n = V1 / torch.norm(V1, dim=0, keepdim=True)

    plt.subplot(141)
    plt.imshow((U1n.T @ U1n).cpu().numpy(), cmap="viridis_r", vmin=-1, vmax=1)
    plt.colorbar()
    plt.axis("off")

    plt.subplot(142)
    plt.imshow((V1n.T @ V1n).cpu().numpy(), cmap="viridis_r", vmin=-1, vmax=1)
    plt.colorbar()
    plt.axis("off")

    # ---- Model2 U/V ----
    U2, V2 = out2["U"][0], out2["V"][0]
    U2n = U2 / torch.norm(U2, dim=0, keepdim=True)
    V2n = V2 / torch.norm(V2, dim=0, keepdim=True)

    plt.subplot(143)
    plt.imshow((U2n.T @ U2n).cpu().numpy(), cmap="viridis_r", vmin=-1, vmax=1)
    plt.colorbar()
    plt.axis("off")

    plt.subplot(144)
    plt.imshow((V2n.T @ V2n).cpu().numpy(), cmap="viridis_r", vmin=-1, vmax=1)
    plt.colorbar()
    plt.axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(path, "orthogonality.png"))
    plt.close()

    # ==========================================================
    # 3. Save MSE, RelErr and OrthErr to txt
    # ==========================================================
    def orth_err_normalized(X):
        # Column normalization
        Xn = F.normalize(X, p=2, dim=0)
        r = Xn.shape[1]
        I = torch.eye(r, device=X.device)
        return torch.norm(Xn.T @ Xn - I, p="fro") / torch.norm(I, p="fro")

    # Orthogonality errors
    U1_err, V1_err = orth_err_normalized(U1n).item(), orth_err_normalized(V1n).item()
    U2_err, V2_err = orth_err_normalized(U2n).item(), orth_err_normalized(V2n).item()

    # Relative error (ratio to SVD)
    model1_relerr = model1_mse / svd_mse
    model2_relerr = model2_mse / svd_mse

    with open(os.path.join(path, "mse.txt"), "w") as f:
        f.write(f"SVD MSE:          {svd_mse:.6f}\n")
        f.write(f"Model1 MSE:       {model1_mse:.6f}\n")
        f.write(f"Model1 RelErr:    {model1_relerr:.6f}\n")
        f.write(f"Model1 OrthErr U: {U1_err:.6f}, V: {V1_err:.6f}\n")
        f.write(f"Model2 MSE:       {model2_mse:.6f}\n")
        f.write(f"Model2 RelErr:    {model2_relerr:.6f}\n")
        f.write(f"Model2 OrthErr U: {U2_err:.6f}, V: {V2_err:.6f}\n")


visualize_results(data, model1, model2, k, save_path)
